# Reproducibility Audit

## Scientific objective
Verify artifact completeness, checksums, split leakage constraints, seeds, notebook ordering, model-bundle metadata, and deterministic inference.

## Inputs
- Entire generated repository

## Expected outputs
- `reports/reproducibility_audit.json`
- `data/metadata/artifact_manifest.csv`

## Dependencies
Python standard library, pandas, joblib

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
The audit verifies reproducibility infrastructure, not scientific validity or external generalization.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
Exact bitwise neural reproducibility may differ across hardware/library builds; deviations must be recorded.

## Next notebook
[25_manuscript_tables_and_figures.ipynb](./25_manuscript_tables_and_figures.ipynb)

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'full', 'seed': 20260723}


In [3]:
from toxicity_screening.utils import sha256_file, atomic_write_json
from toxicity_screening.splitting import assert_no_group_leakage
from toxicity_screening.inference import load_bundles, predict_smiles
required=[ROOT/"data/metadata/dataset_registry.csv",ROOT/"data/processed/split_assignments.csv",ROOT/"results/metrics/qsar_baselines.csv",ROOT/"models/calibrated"]
require_paths(required)

records = pd.read_parquet(
    ROOT / "data/processed/modeling_records.parquet"
)
assert_no_group_leakage(records,"molecule_id","scaffold_split"); assert_no_group_leakage(records,"scaffold","scaffold_split")
files=[p for p in ROOT.rglob('*') if p.is_file() and '.git' not in p.parts and p.suffix not in {'.pyc'}]
manifest=pd.DataFrame({"path":[str(p.relative_to(ROOT)) for p in files],"size_bytes":[p.stat().st_size for p in files],"sha256":[sha256_file(p) for p in files]})
manifest.to_csv(ROOT/"data/metadata/artifact_manifest.csv",index=False)
# Deterministic classical inference check.
bundles=load_bundles(ROOT/"models/calibrated"); a=predict_smiles("CCO",bundles); b=predict_smiles("CCO",bundles)
prob_equal=bool(np.allclose(a.calibrated_probability,b.calibrated_probability,equal_nan=True))
audit={"profile":PROFILE,"files":len(files),"dataset_registry":True,"no_molecule_leakage":True,"no_scaffold_leakage":True,"deterministic_inference":prob_equal,"limitations":"Scientific validity still requires full repeated and external evaluation."}
atomic_write_json(audit,ROOT/"reports/reproducibility_audit.json"); assert prob_equal; audit

{'profile': 'full',
 'files': 731,
 'dataset_registry': True,
 'no_molecule_leakage': True,
 'no_scaffold_leakage': True,
 'deterministic_inference': True,
 'limitations': 'Scientific validity still requires full repeated and external evaluation.'}

### Completion gate
Confirm that the declared artifacts exist before continuing to `25_manuscript_tables_and_figures.ipynb`.